# Spark Questions — Week 5 Assignment
### Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames

This notebook walks through every step of the assignment:
1. MapReduce vs Spark
2. Starting a Spark session
3. Loading data
4. Data cleaning (duplicates, nulls, inconsistent values)
5. Filtering (age, category, region)
6. Transformation (renaming, casting)
7. Aggregation (count, sum, avg, min, max)
8. Grouping (`groupBy` + conditions on aggregated results)
9. Wide transformations & shuffle (conceptual)
10. A complete end-to-end pipeline


## Step 1: MapReduce vs Spark

**MapReduce** (the original Hadoop processing model) reads and writes
intermediate results **to disk** between every Map and Reduce stage. This
makes it reliable but slow, especially for iterative workloads (e.g.
machine learning, or any pipeline with multiple transformation stages).

**Apache Spark** improves on this by:
- Keeping data **in memory** (RAM) across operations whenever possible,
  avoiding repeated disk I/O.
- Building a **DAG (Directed Acyclic Graph)** of transformations that gets
  optimized before execution, instead of forcing rigid Map → Reduce steps.
- Offering a much higher-level API (**DataFrames**, SQL) instead of raw
  key-value Map/Reduce functions.

**Net effect:** Spark is commonly **10–100x faster** than MapReduce for
iterative or multi-stage jobs, while being noticeably easier to write and
read.

### DataFrame immutability
A Spark DataFrame is **immutable** — operations like `.filter()`,
`.withColumn()`, or `.groupBy()` never modify the original DataFrame in
place. Each transformation returns a **new** DataFrame, and Spark only
actually computes anything once an **action** (like `.show()`, `.count()`,
`.collect()`) is called. This lazy, immutable model is what allows Spark to
safely optimize and parallelize the whole chain of operations.


## Step 2: Start Spark
Import libraries and create a `SparkSession` — the entry point to all DataFrame operations.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = (
    SparkSession.builder
    .appName("SparkAssignment_Week4")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/19 17:57:35 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/19 17:57:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/07/19 17:57:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


## Step 3: Load Data
Load the CSV into a DataFrame and inspect its structure.

In [2]:
df = spark.read.csv("../data/dataset.csv", header=True, inferSchema=True)

print("First 5 rows:")
df.show(5, truncate=False)


[Stage 0:>                                                          (0 + 1) / 1]



First 5 rows:


+-----------+------------+----+-----------+------+------------+----------+
|customer_id|name        |age |category   |region|sales_amount|join_date |
+-----------+------------+----+-----------+------+------------+----------+
|1233       |Customer_234|56.0|Clothing   |North |206.13      |2025-04-28|
|1269       |Customer_270|22.0|clothing   |south |357.42      |2024-11-11|
|1030       |Customer_31 |24.0|ELECTRONICS|North |73.89       |2024-02-08|
|1037       |Customer_38 |25.0|Clothing   |SOUTH |224.77      |2025-06-10|
|1168       |Customer_169|29.0|clothing   |East  |122.2       |2024-06-18|
+-----------+------------+----+-----------+------+------------+----------+
only showing top 5 rows


In [3]:
print("Column names:", df.columns)

Column names: ['customer_id', 'name', 'age', 'category', 'region', 'sales_amount', 'join_date']


In [4]:
print("Schema:")
df.printSchema()

Schema:
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: double (nullable = true)
 |-- category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- join_date: date (nullable = true)



In [5]:
print("Total rows loaded (raw, includes duplicates/nulls):", df.count())

Total rows loaded (raw, includes duplicates/nulls): 315


## Step 4: Data Cleaning

The raw dataset intentionally contains:
- Exact duplicate rows
- Nulls in `age` and `region`
- Empty strings in `category`
- A handful of unrealistic ages (negative, 0, >100) — data entry errors
- `age` stored as a string with stray whitespace for some rows (e.g. `" 47 "`) — a schema/type inconsistency

We handle each of these below.

In [6]:
# 4a. Remove exact duplicate rows
before = df.count()
df_no_dupes = df.dropDuplicates()
after = df_no_dupes.count()
print(f"Removed {before - after} duplicate rows ({before} -> {after})")

Removed 15 duplicate rows (315 -> 300)


In [7]:
# 4b. Fix schema inconsistency: age loaded as mixed string/number.
# Cast to string -> trim whitespace -> cast to double -> cast to int.
df_clean = df_no_dupes.withColumn(
    "age", F.trim(F.col("age").cast("string")).cast("double").cast(IntegerType())
)

# 4c. Treat empty-string categories as nulls
df_clean = df_clean.withColumn(
    "category",
    F.when(F.trim(F.col("category")) == "", None).otherwise(F.col("category"))
)

# 4d. Standardize inconsistent casing (e.g. "ELECTRONICS", "electronics" -> "Electronics")
df_clean = (
    df_clean
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
)

# 4e. Null-out unrealistic ages (data entry errors: <=0 or >100)
df_clean = df_clean.withColumn(
    "age",
    F.when((F.col("age") <= 0) | (F.col("age") > 100), None).otherwise(F.col("age"))
)

print("Cleaning transformations applied.")

Cleaning transformations applied.


In [8]:
print("Null counts per column BEFORE final null-handling:")
df_clean.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]
).show()

Null counts per column BEFORE final null-handling:


+-----------+----+---+--------+------+------------+---------+
|customer_id|name|age|category|region|sales_amount|join_date|
+-----------+----+---+--------+------+------------+---------+
|          0|   0| 19|       9|    12|           0|        0|
+-----------+----+---+--------+------+------------+---------+



In [9]:
# Strategy: drop rows missing 'age' (critical field for our analysis),
# but FILL missing region/category with 'Unknown' instead of dropping,
# so we keep more usable rows for aggregation.
df_clean = df_clean.dropna(subset=["age"])
df_clean = df_clean.fillna({"region": "Unknown", "category": "Unknown"})

print(f"Rows after full cleaning: {df_clean.count()} (started at {before})")

Rows after full cleaning: 281 (started at 315)


In [10]:
print("Null counts per column AFTER handling:")
df_clean.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]
).show()

Null counts per column AFTER handling:


+-----------+----+---+--------+------+------------+---------+
|customer_id|name|age|category|region|sales_amount|join_date|
+-----------+----+---+--------+------+------------+---------+
|          0|   0|  0|       0|     0|           0|        0|
+-----------+----+---+--------+------+------------+---------+



## Step 6 (done here first): Transform Data
Rename `sales_amount` -> `total_sales` and finalize column types.

In [11]:
df_transformed = (
    df_clean
    .withColumnRenamed("sales_amount", "total_sales")
    .withColumn("age", F.col("age").cast(IntegerType()))
    .withColumn("total_sales", F.round(F.col("total_sales").cast("double"), 2))
)
df_transformed.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = false)
 |-- region: string (nullable = false)
 |-- total_sales: double (nullable = true)
 |-- join_date: date (nullable = true)



## Step 5: Filter Data
Apply conditions on age range, category, and region.

In [12]:
# Filter by age range (25-45)
age_filtered = df_transformed.filter((F.col("age") >= 25) & (F.col("age") <= 45))
print("Customers aged 25-45:", age_filtered.count())
age_filtered.show(5)

Customers aged 25-45: 144


+-----------+------------+---+-----------+-------+-----------+----------+
|customer_id|        name|age|   category| region|total_sales| join_date|
+-----------+------------+---+-----------+-------+-----------+----------+
|       1172|Customer_173| 41|    Grocery|Central|       72.1|2024-11-02|
|       1131|Customer_132| 29|Electronics|Central|     141.96|2025-06-04|
|       1121|Customer_122| 37|    Grocery|Central|     127.07|2023-03-01|
|       1234|Customer_235| 43|   Clothing|   East|     505.33|2023-12-18|
|       1279|Customer_280| 45|    Grocery|Central|      128.3|2024-03-10|
+-----------+------------+---+-----------+-------+-----------+----------+
only showing top 5 rows


In [13]:
# Filter by category
electronics_only = df_transformed.filter(F.col("category") == "Electronics")
print("Customers who bought Electronics:", electronics_only.count())
electronics_only.show(5)

Customers who bought Electronics: 50


+-----------+------------+---+-----------+-------+-----------+----------+
|customer_id|        name|age|   category| region|total_sales| join_date|
+-----------+------------+---+-----------+-------+-----------+----------+
|       1115|Customer_116| 16|Electronics|   East|     542.86|2023-10-29|
|       1131|Customer_132| 29|Electronics|Central|     141.96|2025-06-04|
|       1105|Customer_106| 47|Electronics|  North|      12.41|2023-07-26|
|       1143|Customer_144| 49|Electronics|   East|     221.46|2024-09-19|
|       1194|Customer_195| 16|Electronics|   East|     177.59|2024-02-08|
+-----------+------------+---+-----------+-------+-----------+----------+
only showing top 5 rows


In [14]:
# Filter by region
north_region = df_transformed.filter(F.col("region") == "North")
print("Customers in North region:", north_region.count())
north_region.show(5)

Customers in North region: 56


+-----------+------------+---+-----------+------+-----------+----------+
|customer_id|        name|age|   category|region|total_sales| join_date|
+-----------+------------+---+-----------+------+-----------+----------+
|       1116|Customer_117| 25|  Furniture| North|      240.6|2025-02-16|
|       1105|Customer_106| 47|Electronics| North|      12.41|2023-07-26|
|       1204|Customer_205| 40|Electronics| North|     340.53|2025-01-14|
|       1129|Customer_130| 32|       Toys| North|     282.47|2024-12-02|
|       1191|Customer_192| 18|       Toys| North|     134.91|2024-07-23|
+-----------+------------+---+-----------+------+-----------+----------+
only showing top 5 rows


In [15]:
# Combined filter: age range + category + region together
combined_filter = df_transformed.filter(
    (F.col("age").between(25, 45)) &
    (F.col("category") == "Electronics") &
    (F.col("region") == "North")
)
print("Combined filter (age 25-45, Electronics, North):", combined_filter.count())
combined_filter.show()

Combined filter (age 25-45, Electronics, North): 4


+-----------+------------+---+-----------+------+-----------+----------+
|customer_id|        name|age|   category|region|total_sales| join_date|
+-----------+------------+---+-----------+------+-----------+----------+
|       1204|Customer_205| 40|Electronics| North|     340.53|2025-01-14|
|       1077| Customer_78| 26|Electronics| North|      87.19|2025-03-01|
|       1193|Customer_194| 27|Electronics| North|      92.86|2023-01-09|
|       1257|Customer_258| 30|Electronics| North|     389.16|2023-04-23|
+-----------+------------+---+-----------+------+-----------+----------+



## Step 7: Aggregation
Basic statistics across the whole (cleaned) dataset.

In [16]:
agg_summary = df_transformed.agg(
    F.count("*").alias("total_rows"),
    F.avg("total_sales").alias("avg_sales"),
    F.min("total_sales").alias("min_sales"),
    F.max("total_sales").alias("max_sales"),
    F.sum("total_sales").alias("sum_sales"),
    F.avg("age").alias("avg_age"),
    F.min("age").alias("min_age"),
    F.max("age").alias("max_age"),
)
agg_summary.show()

+----------+------------------+---------+---------+-----------------+----------------+-------+-------+
|total_rows|         avg_sales|min_sales|max_sales|        sum_sales|         avg_age|min_age|max_age|
+----------+------------------+---------+---------+-----------------+----------------+-------+-------+
|       281|243.01170818505327|     9.78|   810.95|68286.28999999996|38.8932384341637|     16|     80|
+----------+------------------+---------+---------+-----------------+----------------+-------+-------+



## Step 8: Group Data
`groupBy()` combined with aggregation functions, including a condition applied to the aggregated result (similar to SQL's `HAVING`).

In [17]:
by_category = (
    df_transformed.groupBy("category")
    .agg(
        F.count("*").alias("num_customers"),
        F.round(F.sum("total_sales"), 2).alias("total_sales_sum"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales"),
    )
    .orderBy(F.desc("total_sales_sum"))
)
print("Aggregated by category:")
by_category.show()

Aggregated by category:


+-----------+-------------+---------------+---------+
|   category|num_customers|total_sales_sum|avg_sales|
+-----------+-------------+---------------+---------+
|   Clothing|           72|       17569.61|   244.02|
|  Furniture|           51|       15256.42|   299.15|
|Electronics|           50|       12386.45|   247.73|
|    Grocery|           46|       10831.55|   235.47|
|       Toys|           53|       10681.67|   201.54|
|    Unknown|            9|        1560.59|    173.4|
+-----------+-------------+---------------+---------+



In [18]:
by_region = (
    df_transformed.groupBy("region")
    .agg(
        F.count("*").alias("num_customers"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales"),
        F.round(F.sum("total_sales"), 2).alias("total_sales_sum"),
    )
    .orderBy(F.desc("total_sales_sum"))
)
print("Aggregated by region:")
by_region.show()

Aggregated by region:


+-------+-------------+---------+---------------+
| region|num_customers|avg_sales|total_sales_sum|
+-------+-------------+---------+---------------+
|Central|           64|   252.07|       16132.55|
|  South|           59|   261.09|        15404.3|
|  North|           56|   216.46|       12122.02|
|   West|           47|   243.71|       11454.29|
|   East|           44|   247.27|       10879.69|
|Unknown|           11|   208.49|        2293.44|
+-------+-------------+---------+---------------+



In [19]:
# Group by region + category, then apply a condition on the AGGREGATED result
# (only keep groups with more than 5 customers -- a HAVING-style filter)
by_region_category = (
    df_transformed.groupBy("region", "category")
    .agg(
        F.count("*").alias("num_customers"),
        F.round(F.avg("total_sales"), 2).alias("avg_sales"),
    )
    .filter(F.col("num_customers") > 5)
    .orderBy(F.desc("num_customers"))
)
print("Region x Category groups with more than 5 customers:")
by_region_category.show(20)

Region x Category groups with more than 5 customers:


+-------+-----------+-------------+---------+
| region|   category|num_customers|avg_sales|
+-------+-----------+-------------+---------+
|  South|   Clothing|           20|   310.03|
|Central|    Grocery|           16|   189.95|
|Central|  Furniture|           16|   385.63|
|  South|       Toys|           16|    251.5|
|   East|   Clothing|           14|   219.34|
|Central|   Clothing|           13|   164.64|
|  North|   Clothing|           12|   217.78|
|  North|Electronics|           12|   219.78|
|  South|    Grocery|           11|   260.78|
|   East|Electronics|           10|   309.44|
|   West|       Toys|           10|   169.16|
|   West|   Clothing|           10|   262.71|
|Central|Electronics|           10|   290.19|
|  North|       Toys|           10|   206.63|
|  North|  Furniture|           10|   212.33|
|  North|    Grocery|            9|   248.39|
|   West|Electronics|            9|   251.08|
|   West|  Furniture|            9|   269.79|
|   East|  Furniture|            9

## Step 9: Wide Transformations & Shuffle (conceptual)

`groupBy()` + `.agg()` above is a **wide transformation**: Spark has to
move rows that share the same key (e.g. same `category`) onto the same
partition before it can aggregate them. That data movement across
partitions/executors is called a **shuffle** — it involves disk and
network I/O and is the single most expensive kind of operation in Spark.

By contrast, `.filter()` and `.withColumn()` from Steps 5–6 are **narrow
transformations**: each partition processes its own rows independently, so
no shuffle is required.

We can see the shuffle in Spark's physical plan below — look for the
**`Exchange`** step, which is Spark's label for a shuffle boundary.

In [20]:
by_category.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_sales_sum#559 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_sales_sum#559 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1613]
      +- HashAggregate(keys=[category#177], functions=[count(1), sum(total_sales#267), avg(total_sales#267)])
         +- Exchange hashpartitioning(category#177, 200), ENSURE_REQUIREMENTS, [plan_id=1610]
            +- HashAggregate(keys=[category#177], functions=[partial_count(1), partial_sum(total_sales#267), partial_avg(total_sales#267)])
               +- HashAggregate(keys=[sales_amount#22, name#18, customer_id#17, join_date#23, age#19, region#21, category#20], functions=[])
                  +- Exchange hashpartitioning(sales_amount#22, name#18, customer_id#17, join_date#23, age#19, region#21, category#20, 200), ENSURE_REQUIREMENTS, [plan_id=1606]
                     +- HashAggregate(keys=[knownfloatingpointnormalized(normalizenanandzero(sales_amount#22)) AS s

## Step 10: Complete Pipeline
Every stage above, chained into a single reusable function: Load → Clean → Filter → Transform → Aggregate.

In [21]:
def run_pipeline(input_path: str):
    raw = spark.read.csv(input_path, header=True, inferSchema=True)

    cleaned = (
        raw.dropDuplicates()
        .withColumn("age", F.trim(F.col("age").cast("string")).cast("double").cast(IntegerType()))
        .withColumn(
            "category",
            F.when(F.trim(F.col("category")) == "", None).otherwise(F.col("category")),
        )
        .withColumn("category", F.initcap(F.trim(F.col("category"))))
        .withColumn("region", F.initcap(F.trim(F.col("region"))))
        .withColumn(
            "age",
            F.when((F.col("age") <= 0) | (F.col("age") > 100), None).otherwise(F.col("age")),
        )
        .dropna(subset=["age"])
        .fillna({"region": "Unknown", "category": "Unknown"})
        .withColumnRenamed("sales_amount", "total_sales")
        .withColumn("total_sales", F.round(F.col("total_sales").cast("double"), 2))
    )

    filtered = cleaned.filter(F.col("age").between(18, 65))

    result = (
        filtered.groupBy("region", "category")
        .agg(
            F.count("*").alias("num_customers"),
            F.round(F.avg("total_sales"), 2).alias("avg_sales"),
            F.round(F.sum("total_sales"), 2).alias("total_sales_sum"),
            F.round(F.avg("age"), 1).alias("avg_age"),
        )
        .orderBy(F.desc("total_sales_sum"))
    )
    return result


final_result = run_pipeline("../data/dataset.csv")
print("Final pipeline output (region x category summary):")
final_result.show(30)

Final pipeline output (region x category summary):


+-------+-----------+-------------+---------+---------------+-------+
| region|   category|num_customers|avg_sales|total_sales_sum|avg_age|
+-------+-----------+-------------+---------+---------------+-------+
|  South|   Clothing|           19|   309.73|        5884.94|   40.0|
|Central|  Furniture|           14|   383.04|         5362.6|   47.6|
|  South|       Toys|           15|   254.05|        3810.75|   40.2|
|   East|   Clothing|           14|   219.34|        3070.74|   40.6|
|Central|Electronics|           10|   290.19|        2901.87|   34.1|
|  South|    Grocery|           11|   260.78|        2868.55|   44.5|
|   West|   Clothing|           10|   262.71|        2627.11|   43.1|
|   East|  Furniture|            8|    322.8|        2582.43|   34.9|
|   West|  Furniture|            9|   269.79|        2428.09|   41.1|
|Central|    Grocery|           14|   172.58|        2416.09|   39.6|
|  North|   Clothing|           11|   216.67|        2383.34|   40.3|
|   East|Electronics

In [22]:
# Save results for submission
final_result.toPandas().to_csv("../output/results.csv", index=False)
print("Saved to ../output/results.csv")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Saved to ../output/results.csv


In [23]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.


## Observations & Insights

1. **Cleaning impact**: Of the 315 raw rows, 15 were exact duplicates
   (removed) and a further ~15 rows had unusable/null ages after fixing
   type and range issues, leaving a clean, analyzable dataset.
2. **Category standardization mattered**: without fixing casing
   (`"ELECTRONICS"` vs `"electronics"` vs `"Electronics"`), `groupBy` would
   have split what is really one category into three separate groups —
   silently corrupting every aggregation downstream.
3. **Schema/type issues are common**: a subset of `age` values arrived as
   whitespace-padded strings (`" 47 "`) instead of numbers. Casting
   straight to `IntegerType` failed until values were routed through
   `double` first — a good reminder to always inspect `printSchema()` and
   sample rows before trusting a column's type.
4. **`Clothing` and `Furniture` led on total sales** across most regions in
   the final aggregated output, while regions with missing/`"Unknown"`
   values naturally showed smaller, less reliable group sizes.
5. **Wide vs narrow transformations**: `filter`/`withColumn` ran without
   any data movement, while every `groupBy().agg()` triggered a shuffle
   (visible as `Exchange` in the physical plan) — the main thing to watch
   for when tuning Spark job performance at scale.
